# 05 features y modelos intermedios



### Modelo 9 — Más características del historial

Calculamos nuevas variables de atrasos y movimientos recientes a partir de los registros bancarios previos a cada solicitud. Volvimos a entrenar y comparar la ganancia en validación. La idea era comprobar si resumir de distintas formas el historial aportaba información útil sin cambiar el algoritmo de base.

In [ ]:

# armamos nuevas variables a partir del historial de cada solicitud
def crear_features_extra(historial):
    datos = historial.sort_values(
        ["ID_Solicitud", "Mes_Referencia"]
    ).copy()

    # resumimos los atrasos y la variacion del saldo
    features = datos.groupby("ID_Solicitud").agg(
        Meses_Con_Atraso=(
            "Dias_Atraso_Otras_Deudas",
            lambda x: (x > 0).sum()
        ),
        Meses_Atraso_30=(
            "Dias_Atraso_Otras_Deudas",
            lambda x: (x >= 30).sum()
        ),
        Saldo_Desvio=(
            "Saldo_Promedio_Mensual", "std"
        ),
        Saldo_Ultimo=(
            "Saldo_Promedio_Mensual", "last"
        )
    )

    # tomamos los ultimos tres registros de cada solicitud
    ultimos_3 = datos.groupby(
        "ID_Solicitud"
    ).tail(3)

    recientes = ultimos_3.groupby("ID_Solicitud").agg(
        Saldo_Ultimos_3=(
            "Saldo_Promedio_Mensual", "mean"
        ),
        Atraso_Ultimos_3=(
            "Dias_Atraso_Otras_Deudas", "mean"
        )
    )

    features = features.join(recientes)

    # reemplazamos el desvio faltante si hay un solo registro
    features["Saldo_Desvio"] = features["Saldo_Desvio"].fillna(0)

    return features.reset_index()


# creamos las caracteristicas para entrenamiento y test
features_extra_train = crear_features_extra(historial)
features_extra_test = crear_features_extra(historial_test)

print("Train:", features_extra_train.shape)
print("Test:", features_extra_test.shape)
display(features_extra_train.head())


In [ ]:

# agregamos las nuevas variables sin modificar los datasets anteriores
train_fe = train_bancario.merge(
    features_extra_train,
    on="ID_Solicitud",
    how="left",
    validate="one_to_one"
)

test_fe = test_bancario.merge(
    features_extra_test,
    on="ID_Solicitud",
    how="left",
    validate="one_to_one"
)

# creamos relaciones entre monto, saldo e ingresos
for df in [train_fe, test_fe]:
    df["Monto_Sobre_Saldo"] = (
        df["Monto_Solicitado"] /
        (df["Saldo_Promedio"].clip(lower=0) + 100)
    )

    df["Saldo_Sobre_Ingreso"] = (
        df["Saldo_Promedio"] /
        df["Ingreso_Mensual"].clip(lower=1)
    )

    df["Cambio_Saldo"] = (
        df["Saldo_Ultimos_3"] - df["Saldo_Promedio"]
    )

    df["Frecuencia_Atraso"] = (
        df["Meses_Con_Atraso"] /
        df["Meses_Observados"].clip(lower=1)
    )

# juntamos las variables anteriores con las nuevas
columnas_fe = columnas_modelo + [
    "Meses_Con_Atraso",
    "Meses_Atraso_30",
    "Saldo_Desvio",
    "Saldo_Ultimo",
    "Saldo_Ultimos_3",
    "Atraso_Ultimos_3",
    "Monto_Sobre_Saldo",
    "Saldo_Sobre_Ingreso",
    "Cambio_Saldo",
    "Frecuencia_Atraso"
]

# comprobamos que las variables esten completas
assert train_fe[columnas_fe].notna().all().all()
assert test_fe[columnas_fe].notna().all().all()

print("Cantidad de variables:", len(columnas_fe))
print("Train:", train_fe[columnas_fe].shape)
print("Test:", test_fe[columnas_fe].shape)


In [ ]:

# usamos la misma division temporal de los modelos anteriores
X_train_fe = train_fe.loc[
    entrenamiento_temporal, columnas_fe
]

X_val_fe = train_fe.loc[
    validacion_temporal, columnas_fe
]

# entrenamos la regresion logistica con las 19 variables
modelo_fe = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

modelo_fe.fit(X_train_fe, y_train_temp)

# calculamos las predicciones de validacion
pred_fe = modelo_fe.predict(X_val_fe)
prob_fe = modelo_fe.predict_proba(X_val_fe)[:, 1]

print("Accuracy:", accuracy_score(y_val_temp, pred_fe))
print("\nMatriz de confusion:")
print(confusion_matrix(y_val_temp, pred_fe))
print("\nReporte:")
print(classification_report(y_val_temp, pred_fe))


In [ ]:

# calculamos la ganancia del modelo con nuevas variables
cartera_fe = evaluar_cartera(
    train_fe.loc[X_val_fe.index],
    prob_fe
)

ganancia_fe = cartera_fe["Ganancia_Real"].sum()

# comparamos con los modelos anteriores
comparacion_fe = pd.DataFrame({
    "Modelo": [
        "Logistica con 9 variables",
        "Random Forest",
        "Logistica con 19 variables"
    ],
    "Ganancia": [
        608210.86,
        511974.89,
        ganancia_fe
    ]
})

display(
    comparacion_fe.sort_values(
        "Ganancia", ascending=False
    )
)

print("Diferencia contra Modelo 5:", round(
    ganancia_fe - 608210.86, 2
))


In [ ]:

# generamos una nueva submission si mejora la ganancia temporal
if ganancia_fe > 608210.86:

    # entrenamos el modelo final con todos los datos
    modelo_fe_final = Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    modelo_fe_final.fit(
        train_fe[columnas_fe],
        train_fe["Estado_Final"]
    )

    # calculamos las probabilidades de test
    prob_fe_test = modelo_fe_final.predict_proba(
        test_fe[columnas_fe]
    )[:, 1]

    # armamos la cartera y calculamos el valor esperado
    cartera_fe_test = test_fe[
        ["ID_Solicitud", "Monto_Solicitado", "Tasa_Interes"]
    ].copy()

    cartera_fe_test["EV"] = (
        cartera_fe_test["Monto_Solicitado"] *
        (
            prob_fe_test * cartera_fe_test["Tasa_Interes"]
            - (1 - prob_fe_test)
        )
    )

    cartera_fe_test = cartera_fe_test.sort_values(
        "EV", ascending=False
    )

    # aprobamos los prestamos sin superar los 2 millones
    cartera_fe_test["Decision_Aprobacion"] = 0
    capital_fe = 0

    for indice, fila in cartera_fe_test.iterrows():
        monto = fila["Monto_Solicitado"]

        if fila["EV"] > 0 and capital_fe + monto <= 2_000_000:
            cartera_fe_test.loc[indice, "Decision_Aprobacion"] = 1
            capital_fe += monto

    # generamos el archivo con las decisiones en el orden original
    submission_v4 = test[["ID_Solicitud"]].merge(
        cartera_fe_test[["ID_Solicitud", "Decision_Aprobacion"]],
        on="ID_Solicitud",
        how="left",
        validate="one_to_one"
    )

    # revisamos que el archivo respete las condiciones
    assert len(submission_v4) == len(test)
    assert submission_v4["Decision_Aprobacion"].notna().all()
    assert submission_v4["Decision_Aprobacion"].isin([0, 1]).all()
    assert capital_fe <= 2_000_000

    submission_v4.to_csv("submission_v4.csv", index=False)

    print("Prestamos aprobados:",
          submission_v4["Decision_Aprobacion"].sum())
    print("Capital utilizado:", capital_fe)

    print("Archivo disponible para descargar:", "submission_v4.csv")
else:
    print("El nuevo modelo no supero al Modelo 5.")


### Modelo 10 — Selección de características

Comparamos distintos conjuntos de variables bancarias y financieras para ver cuáles convenía conservar. No usamos solo una métrica de clasificación: también revisamos la ganancia de la cartera. Esta comparación ayudó a definir el grupo de características que utilizamos en pruebas posteriores.

In [ ]:

# definimos los grupos de variables que queremos probar
grupos_features = {
    "Modelo 5 - Original": columnas_modelo,

    "Atrasos": columnas_modelo + [
        "Meses_Con_Atraso",
        "Meses_Atraso_30",
        "Atraso_Ultimos_3",
        "Frecuencia_Atraso"
    ],

    "Saldos": columnas_modelo + [
        "Saldo_Desvio",
        "Saldo_Ultimo",
        "Saldo_Ultimos_3",
        "Cambio_Saldo"
    ],

    "Relaciones financieras": columnas_modelo + [
        "Monto_Sobre_Saldo",
        "Saldo_Sobre_Ingreso"
    ],

    "Todas las variables": columnas_fe
}

resultados_features = []

# entrenamos una regresion logistica para cada grupo
for nombre, columnas in grupos_features.items():

    X_ent = train_fe.loc[
        entrenamiento_temporal, columnas
    ]

    X_valid = train_fe.loc[
        validacion_temporal, columnas
    ]

    modelo = Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    modelo.fit(X_ent, y_train_temp)

    # calculamos las probabilidades y la ganancia
    probabilidades = modelo.predict_proba(X_valid)[:, 1]

    cartera = evaluar_cartera(
        train_fe.loc[X_valid.index],
        probabilidades
    )

    resultados_features.append({
        "Grupo": nombre,
        "Variables": len(columnas),
        "Accuracy": accuracy_score(
            y_val_temp,
            modelo.predict(X_valid)
        ),
        "Ganancia": cartera["Ganancia_Real"].sum(),
        "Aprobados": int(cartera["Aprobar"].sum())
    })

# mostramos la comparacion de todos los grupos
resultados_features = pd.DataFrame(resultados_features)

display(
    resultados_features.sort_values(
        "Ganancia", ascending=False
    )
)


### Submission V4 — Regresión logística con variables de atrasos

Preparamos una nueva versión con más información bancaria, incluida la morosidad. Comprobamos que la suma de préstamos aprobados respetara el presupuesto y generamos el archivo de Kaggle. La V4 fue nuestra principal referencia durante varios experimentos; más adelante la V7 superó su mejor puntaje público.

In [ ]:

# usamos las variables originales y las nuevas de atrasos
columnas_v4 = columnas_modelo + [
    "Meses_Con_Atraso",
    "Meses_Atraso_30",
    "Atraso_Ultimos_3",
    "Frecuencia_Atraso"
]

# entrenamos con todas las solicitudes historicas
modelo_v4 = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

modelo_v4.fit(
    train_fe[columnas_v4],
    train_fe["Estado_Final"]
)

# calculamos las probabilidades de pago para kaggle
prob_v4 = modelo_v4.predict_proba(
    test_fe[columnas_v4]
)[:, 1]

# calculamos el valor esperado de cada solicitud
cartera_v4 = test_fe[
    ["ID_Solicitud", "Monto_Solicitado", "Tasa_Interes"]
].copy()

cartera_v4["EV"] = (
    cartera_v4["Monto_Solicitado"] *
    (
        prob_v4 * cartera_v4["Tasa_Interes"]
        - (1 - prob_v4)
    )
)

# ordenamos por ganancia esperada
cartera_v4 = cartera_v4.sort_values(
    "EV", ascending=False
)

# seleccionamos prestamos sin superar el presupuesto
cartera_v4["Decision_Aprobacion"] = 0
capital_v4 = 0

for indice, fila in cartera_v4.iterrows():
    monto = fila["Monto_Solicitado"]

    if fila["EV"] > 0 and capital_v4 + monto <= 2_000_000:
        cartera_v4.loc[indice, "Decision_Aprobacion"] = 1
        capital_v4 += monto

# recuperamos el orden original del archivo de test
submission_v4 = test[["ID_Solicitud"]].merge(
    cartera_v4[["ID_Solicitud", "Decision_Aprobacion"]],
    on="ID_Solicitud",
    how="left",
    validate="one_to_one"
)

# comprobamos que el archivo tenga el formato correcto
assert len(submission_v4) == len(test)
assert submission_v4["Decision_Aprobacion"].notna().all()
assert submission_v4["Decision_Aprobacion"].isin([0, 1]).all()
assert capital_v4 <= 2_000_000

# guardamos y descargamos el archivo
submission_v4.to_csv("submission_v4.csv", index=False)

print("Prestamos aprobados:", submission_v4["Decision_Aprobacion"].sum())
print("Capital utilizado:", capital_v4)
print("Presupuesto restante:", 2_000_000 - capital_v4)

print("Archivo disponible para descargar:", "submission_v4.csv")

### Modelo 11 — KNN con características bancarias

Probamos KNN con escalado y distintas configuraciones de vecinos. Medimos la ganancia financiera, no solamente cuántas clasificaciones acertaba. Los resultados no justificaron reemplazar la mejor regresión logística que teníamos, así que seguimos comparando otras alternativas.

In [ ]:

from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# usamos las mismas variables de la submission v4
X_train_knn = train_fe.loc[
    entrenamiento_temporal, columnas_v4
]

X_val_knn = train_fe.loc[
    validacion_temporal, columnas_v4
]

resultados_knn = []

# probamos distintas cantidades de vecinos
for k in [5, 10, 20, 30, 50, 75, 100, 150]:

    modelo_knn = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(
            n_neighbors=k,
            weights="distance",
            n_jobs=-1
        ))
    ])

    modelo_knn.fit(X_train_knn, y_train_temp)

    # calculamos las predicciones
    pred_knn = modelo_knn.predict(X_val_knn)
    prob_knn = modelo_knn.predict_proba(X_val_knn)[:, 1]

    # evaluamos la ganancia de cada configuracion
    cartera_knn = evaluar_cartera(
        train_fe.loc[X_val_knn.index],
        prob_knn
    )

    resultados_knn.append({
        "K": k,
        "Accuracy": accuracy_score(y_val_temp, pred_knn),
        "Ganancia": cartera_knn["Ganancia_Real"].sum(),
        "Aprobados": int(cartera_knn["Aprobar"].sum())
    })

# ordenamos los resultados por ganancia
tabla_knn = pd.DataFrame(resultados_knn)

display(
    tabla_knn.sort_values("Ganancia", ascending=False)
)


In [ ]:

# buscamos la configuracion de knn con mayor ganancia
mejor_knn = tabla_knn.sort_values(
    "Ganancia", ascending=False
).iloc[0]

mejor_k = int(mejor_knn["K"])
mejor_ganancia_knn = mejor_knn["Ganancia"]

# comparamos con el modelo de atrasos
comparacion_knn = pd.DataFrame({
    "Modelo": [
        "Logistica + Atrasos",
        f"KNN (K={mejor_k})"
    ],
    "Ganancia_Temporal": [
        614136.00,
        mejor_ganancia_knn
    ]
})

display(comparacion_knn)

print("Mejor K:", mejor_k)
print(
    "Diferencia:",
    round(mejor_ganancia_knn - 614136.00, 2)
)


In [ ]:

# generamos la submission si knn supera al modelo anterior
if mejor_ganancia_knn > 614136.00:

    # entrenamos knn con todas las solicitudes historicas
    modelo_knn_final = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(
            n_neighbors=mejor_k,
            weights="distance",
            n_jobs=-1
        ))
    ])

    modelo_knn_final.fit(
        train_fe[columnas_v4],
        train_fe["Estado_Final"]
    )

    # calculamos las probabilidades para test
    prob_knn_test = modelo_knn_final.predict_proba(
        test_fe[columnas_v4]
    )[:, 1]

    # calculamos el valor esperado de cada prestamo
    cartera_knn_test = test_fe[
        ["ID_Solicitud", "Monto_Solicitado", "Tasa_Interes"]
    ].copy()

    cartera_knn_test["EV"] = (
        cartera_knn_test["Monto_Solicitado"] *
        (
            prob_knn_test * cartera_knn_test["Tasa_Interes"]
            - (1 - prob_knn_test)
        )
    )

    cartera_knn_test = cartera_knn_test.sort_values(
        "EV", ascending=False
    )

    # seleccionamos los prestamos respetando el presupuesto
    cartera_knn_test["Decision_Aprobacion"] = 0
    capital_knn = 0

    for indice, fila in cartera_knn_test.iterrows():
        monto = fila["Monto_Solicitado"]

        if fila["EV"] > 0 and capital_knn + monto <= 2_000_000:
            cartera_knn_test.loc[indice, "Decision_Aprobacion"] = 1
            capital_knn += monto

    # recuperamos el orden original de las solicitudes
    submission_v5 = test[["ID_Solicitud"]].merge(
        cartera_knn_test[["ID_Solicitud", "Decision_Aprobacion"]],
        on="ID_Solicitud",
        how="left",
        validate="one_to_one"
    )

    # comprobamos el formato y el presupuesto
    assert len(submission_v5) == len(test)
    assert submission_v5["Decision_Aprobacion"].notna().all()
    assert submission_v5["Decision_Aprobacion"].isin([0, 1]).all()
    assert capital_knn <= 2_000_000

    submission_v5.to_csv("submission_v5.csv", index=False)

    print("Prestamos aprobados:", submission_v5["Decision_Aprobacion"].sum())
    print("Capital utilizado:", capital_knn)

    print("Archivo disponible para descargar:", "submission_v5.csv")
else:
    print("KNN no supero al modelo de regresion logistica.")
    print("Conservamos la Submission V4 como referencia.")


### Modelo 12 — Árbol de decisión

Entrenamos árboles de decisión con las variables financieras y bancarias para analizar una alternativa más sencilla de interpretar. Probamos sus resultados económicos con la misma restricción de presupuesto. Después profundizamos esta línea ajustando profundidad y tamaño mínimo de hoja, pero no terminó superando a la regresión elegida.

In [ ]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# usamos las mismas 13 variables de la v4
X_train_arbol = train_fe.loc[
    entrenamiento_temporal, columnas_v4
]

X_val_arbol = train_fe.loc[
    validacion_temporal, columnas_v4
]

resultados_arbol = []

# probamos distintas profundidades y tamaños de hoja
for profundidad in [3, 5, 7, 10, 15]:
    for minimo_hoja in [10, 30, 50]:

        modelo_arbol = DecisionTreeClassifier(
            max_depth=profundidad,
            min_samples_leaf=minimo_hoja,
            random_state=42
        )

        modelo_arbol.fit(X_train_arbol, y_train_temp)

        # calculamos las predicciones de validacion
        pred_arbol = modelo_arbol.predict(X_val_arbol)
        prob_arbol = modelo_arbol.predict_proba(X_val_arbol)[:, 1]

        # calculamos la ganancia con el presupuesto de 2 millones
        cartera_arbol = evaluar_cartera(
            train_fe.loc[X_val_arbol.index],
            prob_arbol
        )

        resultados_arbol.append({
            "Profundidad": profundidad,
            "Min_Hoja": minimo_hoja,
            "Accuracy": accuracy_score(y_val_temp, pred_arbol),
            "Ganancia": cartera_arbol["Ganancia_Real"].sum(),
            "Aprobados": int(cartera_arbol["Aprobar"].sum())
        })

# ordenamos las configuraciones por ganancia
tabla_arbol = pd.DataFrame(resultados_arbol)

display(
    tabla_arbol.sort_values(
        "Ganancia", ascending=False
    )
)


### Modelo 13 — Morosidad reciente

Agregamos indicadores del comportamiento bancario más cercano a cada solicitud, como los atrasos recientes. Buscábamos distinguir clientes que venían empeorando de otros que tenían atrasos antiguos. Comparamos las variantes con la selección financiera anterior; no apareció una mejora suficientemente consistente para reemplazar el modelo de referencia.

In [ ]:

import pandas as pd
import numpy as np

# creamos las variables de morosidad reciente
def crear_morosidad_reciente(historial):
    datos = historial.sort_values(
        ["ID_Solicitud", "Mes_Referencia"]
    ).copy()

    # buscamos los ultimos tres registros de cada solicitud
    ultimos_3 = datos.groupby("ID_Solicitud").tail(3)

    recientes = ultimos_3.groupby("ID_Solicitud").agg(
        Atraso_Max_Ultimos_3=(
            "Dias_Atraso_Otras_Deudas", "max"
        ),
        Meses_Atraso_Reciente=(
            "Dias_Atraso_Otras_Deudas",
            lambda x: (x > 0).sum()
        ),
        Meses_Atraso_30_Reciente=(
            "Dias_Atraso_Otras_Deudas",
            lambda x: (x >= 30).sum()
        ),
        Atraso_Promedio_Reciente=(
            "Dias_Atraso_Otras_Deudas", "mean"
        )
    )

    # calculamos el promedio anterior a los ultimos tres registros
    datos["Posicion"] = datos.groupby(
        "ID_Solicitud"
    ).cumcount()

    datos["Total_Meses"] = datos.groupby(
        "ID_Solicitud"
    )["ID_Solicitud"].transform("size")

    anteriores = datos[
        datos["Posicion"] < datos["Total_Meses"] - 3
    ]

    promedio_anterior = anteriores.groupby(
        "ID_Solicitud"
    )["Dias_Atraso_Otras_Deudas"].mean()

    recientes["Cambio_Atraso"] = (
        recientes["Atraso_Promedio_Reciente"]
        - promedio_anterior.reindex(recientes.index)
    )

    # buscamos la fecha del ultimo atraso de cada solicitud
    meses_con_atraso = datos[
        datos["Dias_Atraso_Otras_Deudas"] > 0
    ]

    ultimo_atraso = meses_con_atraso.groupby(
        "ID_Solicitud"
    )["Mes_Referencia"].max()

    ultimo_atraso = ultimo_atraso.reindex(recientes.index)

    # calculamos los meses desde el ultimo atraso
    fecha_solicitud = datos.groupby(
        "ID_Solicitud"
    )["Fecha_Solicitud"].first().reindex(recientes.index)

    diferencia_meses = (
        fecha_solicitud.dt.year * 12 + fecha_solicitud.dt.month
        - ultimo_atraso.dt.year * 12 - ultimo_atraso.dt.month
    )

    # diferenciamos a los clientes que nunca tuvieron atrasos
    recientes["Nunca_Atraso"] = ultimo_atraso.isna().astype(int)
    recientes["Meses_Desde_Atraso"] = diferencia_meses.fillna(25)

    return recientes.drop(
        columns="Atraso_Promedio_Reciente"
    ).reset_index()


# generamos las variables de entrenamiento y test
morosidad_train = crear_morosidad_reciente(historial)
morosidad_test = crear_morosidad_reciente(historial_test)

# revisamos que se hayan generado correctamente
print("Train:", morosidad_train.shape)
print("Test:", morosidad_test.shape)

print("\nNulos en train:")
print(morosidad_train.isna().sum())

print("\nNulos en test:")
print(morosidad_test.isna().sum())


In [ ]:

# agregamos la nueva informacion sin modificar los modelos anteriores
train_m13 = train_fe.merge(
    morosidad_train,
    on="ID_Solicitud",
    how="left",
    validate="one_to_one"
)

test_m13 = test_fe.merge(
    morosidad_test,
    on="ID_Solicitud",
    how="left",
    validate="one_to_one"
)

# definimos los grupos de variables para comparar
features_recientes = [
    "Atraso_Max_Ultimos_3",
    "Meses_Atraso_Reciente",
    "Meses_Atraso_30_Reciente",
    "Cambio_Atraso",
    "Meses_Desde_Atraso",
    "Nunca_Atraso"
]

grupos_m13 = {
    "V4 - Original": columnas_v4,

    "Atrasos recientes": columnas_v4 + [
        "Atraso_Max_Ultimos_3",
        "Meses_Atraso_Reciente",
        "Meses_Atraso_30_Reciente"
    ],

    "Evolucion de atrasos": columnas_v4 + [
        "Cambio_Atraso",
        "Meses_Desde_Atraso",
        "Nunca_Atraso"
    ],

    "Todas las nuevas": columnas_v4 + features_recientes
}

# comprobamos que las variables esten completas
columnas_nuevas = columnas_v4 + features_recientes

assert train_m13[columnas_nuevas].notna().all().all()
assert test_m13[columnas_nuevas].notna().all().all()

print("Train:", train_m13.shape)
print("Test:", test_m13.shape)


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

resultados_m13 = []

# entrenamos y evaluamos cada grupo de variables
for nombre, columnas in grupos_m13.items():

    X_ent = train_m13.loc[
        entrenamiento_temporal, columnas
    ]

    X_valid = train_m13.loc[
        validacion_temporal, columnas
    ]

    modelo = Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    modelo.fit(X_ent, y_train_temp)

    predicciones = modelo.predict(X_valid)
    probabilidades = modelo.predict_proba(X_valid)[:, 1]

    # calculamos la ganancia de cada configuracion
    cartera = evaluar_cartera(
        train_m13.loc[X_valid.index],
        probabilidades
    )

    resultados_m13.append({
        "Configuracion": nombre,
        "Variables": len(columnas),
        "Accuracy": accuracy_score(y_val_temp, predicciones),
        "Ganancia": cartera["Ganancia_Real"].sum(),
        "Aprobados": int(cartera["Aprobar"].sum())
    })

# ordenamos los resultados por ganancia
tabla_m13 = pd.DataFrame(resultados_m13)

display(
    tabla_m13.sort_values(
        "Ganancia", ascending=False
    )
)


In [ ]:

# buscamos la configuracion con mayor ganancia
mejor_m13 = tabla_m13.sort_values(
    "Ganancia", ascending=False
).iloc[0]

nombre_mejor_m13 = mejor_m13["Configuracion"]
ganancia_mejor_m13 = mejor_m13["Ganancia"]

columnas_mejor_m13 = grupos_m13[nombre_mejor_m13]

print("Mejor configuracion:", nombre_mejor_m13)
print("Cantidad de variables:", len(columnas_mejor_m13))
print("Ganancia:", round(ganancia_mejor_m13, 2))
print(
    "Diferencia contra V4:",
    round(ganancia_mejor_m13 - 614136.00, 2)
)


In [ ]:

# preparamos la submission si mejora la regresion logistica de v4
if ganancia_mejor_m13 > 614136.00:

    # entrenamos el modelo definitivo con todos los datos historicos
    modelo_m13_final = Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    modelo_m13_final.fit(
        train_m13[columnas_mejor_m13],
        train_m13["Estado_Final"]
    )

    # calculamos las probabilidades para las solicitudes nuevas
    prob_m13_test = modelo_m13_final.predict_proba(
        test_m13[columnas_mejor_m13]
    )[:, 1]

    # calculamos el valor esperado de cada prestamo
    cartera_m13_test = test_m13[
        ["ID_Solicitud", "Monto_Solicitado", "Tasa_Interes"]
    ].copy()

    cartera_m13_test["EV"] = (
        cartera_m13_test["Monto_Solicitado"] *
        (
            prob_m13_test * cartera_m13_test["Tasa_Interes"]
            - (1 - prob_m13_test)
        )
    )

    cartera_m13_test = cartera_m13_test.sort_values(
        "EV", ascending=False
    )

    # seleccionamos los prestamos sin superar el presupuesto
    cartera_m13_test["Decision_Aprobacion"] = 0
    capital_m13 = 0

    for indice, fila in cartera_m13_test.iterrows():
        monto = fila["Monto_Solicitado"]

        if fila["EV"] > 0 and capital_m13 + monto <= 2_000_000:
            cartera_m13_test.loc[indice, "Decision_Aprobacion"] = 1
            capital_m13 += monto

    # dejamos las decisiones en el orden original
    submission_v5 = test[["ID_Solicitud"]].merge(
        cartera_m13_test[["ID_Solicitud", "Decision_Aprobacion"]],
        on="ID_Solicitud",
        how="left",
        validate="one_to_one"
    )

    # comprobamos el formato y el presupuesto
    assert len(submission_v5) == len(test)
    assert submission_v5["Decision_Aprobacion"].notna().all()
    assert submission_v5["Decision_Aprobacion"].isin([0, 1]).all()
    assert capital_m13 <= 2_000_000

    submission_v5.to_csv("submission_v5.csv", index=False)

    print("Prestamos aprobados:",
          submission_v5["Decision_Aprobacion"].sum())
    print("Capital utilizado:", capital_m13)

    print("Archivo disponible para descargar:", "submission_v5.csv")
else:
    print("Ninguna configuracion supero a la V4.")
    print("Conservamos el mejor resultado de Kaggle.")


### Modelo 14 — Entrenamiento ponderado

Probamos cambiar el peso de algunos registros durante el entrenamiento de la regresión logística. La intención era comprobar si prestar más atención a determinados ejemplos ayudaba a seleccionar mejor la cartera. Evaluamos las ganancias con el mismo presupuesto y conservamos los resultados como prueba, sin adoptar esta modificación en la entrega final.

In [ ]:

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# usamos las variables de nuestra mejor submission
X_train_pesos = train_fe.loc[
    entrenamiento_temporal, columnas_v4
]

X_val_pesos = train_fe.loc[
    validacion_temporal, columnas_v4
]

# calculamos los pesos segun el monto solicitado
def calcular_pesos(montos, peso_pequenos):
    return np.where(
        montos <= 3000,
        peso_pequenos,
        1.0
    )

resultados_pesos = []

# probamos distintos pesos para los prestamos pequeños
for peso in [1, 1.5, 2, 3, 5]:

    pesos_entrenamiento = calcular_pesos(
        X_train_pesos["Monto_Solicitado"],
        peso
    )

    # entrenamos la regresion logistica usando los pesos
    modelo_pesos = Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    modelo_pesos.fit(
        X_train_pesos,
        y_train_temp,
        modelo__sample_weight=pesos_entrenamiento
    )

    # evaluamos las predicciones sobre los meses posteriores
    pred_pesos = modelo_pesos.predict(X_val_pesos)
    prob_pesos = modelo_pesos.predict_proba(X_val_pesos)[:, 1]

    # calculamos la ganancia con el presupuesto de 2 millones
    cartera_pesos = evaluar_cartera(
        train_fe.loc[X_val_pesos.index],
        prob_pesos
    )

    resultados_pesos.append({
        "Peso": peso,
        "Accuracy": accuracy_score(y_val_temp, pred_pesos),
        "Ganancia": cartera_pesos["Ganancia_Real"].sum(),
        "Aprobados": int(cartera_pesos["Aprobar"].sum())
    })

# ordenamos las configuraciones por ganancia
tabla_pesos = pd.DataFrame(resultados_pesos)

display(
    tabla_pesos.sort_values(
        "Ganancia", ascending=False
    )
)


In [ ]:

# buscamos la ponderacion que dio mayor ganancia
mejor_peso = tabla_pesos.sort_values(
    "Ganancia", ascending=False
).iloc[0]

peso_elegido = float(mejor_peso["Peso"])
ganancia_elegida = float(mejor_peso["Ganancia"])

# comparamos con la mejor configuracion anterior
print("Mejor peso:", peso_elegido)
print("Ganancia:", round(ganancia_elegida, 2))
print(
    "Diferencia contra V4:",
    round(ganancia_elegida - 614136.00, 2)
)
